# M06 — Binary Target and Features

Bus aur Rail ke liye Normal/High target aur historical features prepare karte hain.

## 1. Cleaned data

In [1]:
import numpy as np
import pandas as pd

bus = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_cleaned.csv",
    dtype={"route": str}, parse_dates=["date"]
)
rail = pd.read_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_cleaned.csv",
    dtype={"station_id": str}, parse_dates=["date"]
)

datasets = {
    "bus": bus.rename(columns={"route": "entity_id"}),
    "rail": rail.rename(columns={"station_id": "entity_id"})
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,} cleaned rows")

bus: 167,648 cleaned rows
rail: 209,472 cleaned rows


## 2. Training thresholds

Har route/station ke lower aur upper thresholds 2022 se September 2023 ke data par bante hain. Dono thresholds M07 features ke liye rakhe hain.

In [2]:
thresholds = {}

for name, df in datasets.items():
    if df.empty or df[["entity_id", "date", "rides"]].isna().any().any():
        raise ValueError(f"{name}: cleaned data empty hai ya required values missing hain")
    if df.duplicated(["entity_id", "date"]).any():
        raise ValueError(f"{name}: duplicate ID/date mili hai")

    training = df[df["date"] <= "2023-09-30"]
    groups = training.groupby("entity_id")["rides"]
    limits = groups.quantile([1/3, 2/3]).unstack()
    limits.columns = ["lower", "upper"]
    limits["training_days"] = groups.size()

    eligible = (limits["training_days"] >= 30) & (limits["lower"] < limits["upper"])
    thresholds[name] = limits.loc[eligible].reset_index()
    print(f"{name}: eligible = {eligible.sum()}, excluded = {(~eligible).sum()}")

bus: eligible = 129, excluded = 1
rail: eligible = 141, excluded = 2


## 3. 91-day history aur binary target

Upper threshold se zyada rides = High; baqi = Normal. 91-day history ek simulation choice hai.

In [3]:
binary_data = {}

for name, df in datasets.items():
    history = df[["entity_id", "date", "rides"]].copy()
    history["date"] = history["date"] + pd.Timedelta(days=91)
    history = history.rename(columns={"rides": "lag_91_rides"})

    df = df.merge(history, on=["entity_id", "date"], how="left", validate="one_to_one")
    df = df.merge(thresholds[name], on="entity_id", how="left", validate="many_to_one")

    missing = df["upper"].isna() | df["lag_91_rides"].isna()
    print(f"{name}: excluded rows without thresholds or 91-day history = {missing.sum():,}")
    df = df.loc[~missing].copy()

    df["target"] = np.where(df["rides"] > df["upper"], "High", "Normal")
    df["weekday"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["day_of_month"] = df["date"].dt.day
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df["lag_91_rides"] = df["lag_91_rides"].astype("int64")

    df["split"] = np.select(
        [df["date"].between("2022-01-01", "2023-09-30"),
         df["date"].between("2024-01-01", "2024-12-31"),
         df["date"].between("2025-01-01", "2025-12-31")],
        ["train", "validation", "test"], default="gap"
    )

    columns = ["entity_id", "date", "weekday", "month", "day_of_month", "is_weekend",
               "lag_91_rides", "target", "split"]
    binary_data[name] = df[columns].copy()

    counts = pd.crosstab(df["split"], df["target"]).reindex(
        index=["train", "gap", "validation", "test"], columns=["Normal", "High"], fill_value=0
    )
    print(f"\n{name} — class counts:")
    print(counts.to_string())

C:\Users\muham\AppData\Local\Temp\ipykernel_2372\2846840131.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  history["date"] = history["date"] + pd.Timedelta(days=91)


bus: excluded rows without thresholds or 91-day history = 13,030

bus — class counts:
target      Normal   High
split                    
train        38722  23110
gap           4483   5835
validation   15890  25404
test         13090  28084


C:\Users\muham\AppData\Local\Temp\ipykernel_2372\2846840131.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  history["date"] = history["date"] + pd.Timedelta(days=91)


rail: excluded rows without thresholds or 91-day history = 16,302

rail — class counts:
target      Normal   High
split                    
train        47691  29436
gap           6385   6587
validation   22955  28651
test         19389  32076


Gap rows bhi save hoti hain. Purane split ko use karte waqt inhein train/validation/test mein include na karo; M07 mein har evaluation year ka date cutoff alag hai.

## 4. Save files

Binary model-data files alag hain. Threshold files M07 ki existing files hain aur dobara run karne par overwrite hongi.

In [4]:
for name, df in binary_data.items():
    if not {"train", "validation", "test"}.issubset(set(df["split"])):
        raise ValueError(f"{name}: ek ya zyada splits empty hain")

binary_data["bus"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_binary_model_data.csv", index=False, date_format="%Y-%m-%d"
)
binary_data["rail"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_binary_model_data.csv", index=False, date_format="%Y-%m-%d"
)
thresholds["bus"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\bus_train_thresholds.csv", index=False
)
thresholds["rail"].to_csv(
    r"C:\UZZAM WORK\5th Semester\Artificial Intelligence & Expert Systems\Project\Transit_Ridership_M01_Setup\Transit_Ridership_M10_Reviewed\transit-ridership\backend\data\processed\rail_train_thresholds.csv", index=False
)

for name in ["bus", "rail"]:
    print(f"{name}: saved {len(binary_data[name]):,} binary rows and {len(thresholds[name]):,} thresholds")

bus: saved 154,618 binary rows and 129 thresholds
rail: saved 193,170 binary rows and 141 thresholds
